In [ ]:
import argparse
import configs
import dataloader, baseline_trainer

from configs.eval_fix.configs_yearbook import configs_yearbook_erm
from configs.eval_fix.configs_yearbook import configs_yearbook_swa
from configs.eval_fix.configs_fmow import configs_fmow_erm
from configs.eval_fix.configs_mimic_mortality import configs_mimic_erm
from configs.eval_fix.configs_drug import configs_drug_erm
from configs.eval_fix.configs_huffpost import configs_huffpost_erm
from configs.eval_fix.configs_arxiv import configs_arxiv_erm
import numpy as np
import torch
import torch.nn as nn
import random

from networks.article import ArticleNetwork
from networks.drug import DTI_Encoder, DTI_Classifier
from networks.fmow import FMoWNetwork
from networks.mimic import Transformer
from networks.yearbook import YearbookNetwork
from functools import partial
from methods.agem.agem import AGEM
from methods.coral.coral import DeepCORAL
from methods.erm.erm import ERM
from methods.ewc.ewc import EWC
from methods.ft.ft import FT
from methods.groupdro.groupdro import GroupDRO
from methods.irm.irm import IRM
from methods.si.si import SI
from methods.simclr.simclr import SimCLR
from methods.swa.swa import SWA
from methods.swav.swav import SwaV

import torch
import torch.nn as nn
from lightly.models.modules import SwaVProjectionHead, SwaVPrototypes
from lightly.models.modules.heads import SimCLRProjectionHead

from copy import deepcopy

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import densenet121

IMG_HEIGHT = 224
NUM_CLASSES = 62
method_dict = {'groupdro': 'GroupDRO', 'coral': 'DeepCORAL', 'irm': 'IRM', 'ft': 'FT', 'erm': 'ERM', 'ewc': 'EWC',
                'agem': 'AGEM', 'si': 'SI', 'simclr': 'SimCLR', 'swav': 'SwaV', 'swa': 'SWA'}
class FMoWNetwork2(nn.Module):
    def __init__(self, args, weights=None, ssl_training=False):
        super(FMoWNetwork, self).__init__()
        self.args = args
        self.num_classes = NUM_CLASSES
        self.enc = densenet121(pretrained=True).features
        if args.incor_time:
            hidden_dim = 1025
            self.classifier = nn.Linear(hidden_dim, self.num_classes)
        else:
            hidden_dim = 1024
            self.classifier = nn.Linear(hidden_dim, self.num_classes)
        if weights is not None:
            self.load_state_dict(deepcopy(weights))
        # SimCLR projection head
        if self.args.method == 'simclr':
            from lightly.models.modules.heads import SimCLRProjectionHead
            self.projection_head = SimCLRProjectionHead(hidden_dim, 1024, 128)
        # SwaV: projection head and prototypes
        elif self.args.method == 'swav':
            from lightly.models.modules import SwaVProjectionHead, SwaVPrototypes
            self.projection_head = SwaVProjectionHead(hidden_dim, 1024, 128)
            self.prototypes = SwaVPrototypes(128, n_prototypes=1024)
        self.ssl_training = ssl_training

    def reset_weights(self, weights):
        self.load_state_dict(deepcopy(weights))

    def forward(self, x, t=None):
        features = self.enc(x)
        out = F.relu(features, inplace=True)
        out = F.adaptive_avg_pool2d(out, (1, 1))
        out = torch.flatten(out, 1)
        if t is not None:
            out = torch.cat([out,t],axis=1)
        if self.args.method == 'simclr' and self.ssl_training:
            return self.projection_head(out)
        elif self.args.method == 'swav' and self.ssl_training:
            out = self.projection_head(out)
            out = nn.functional.normalize(out, dim=1, p=2)
            out = self.prototypes(out)
            return out
        else:
            return self.classifier(out)

class YearbookNetwork2(nn.Module):
    def __init__(self, args, num_input_channels, num_classes, ssl_training=False, t=None):
        super(YearbookNetwork2, self).__init__()
        self.args = args
        self.enc = nn.Sequential(self.conv_block(num_input_channels, 32), self.conv_block(32, 32),
                                 self.conv_block(32, 32), self.conv_block(32, 32))
        if args.incor_time:
            self.hid_dim = 33
            #self.classifier = nn.Linear(33, num_classes)
            self.classifier = nn.Linear(33, num_classes)
        else:
            self.hid_dim = 32
            self.classifier = nn.Linear(32, num_classes)            
        # SimCLR: projection head
        if self.args.method == 'simclr':
            self.projection_head = SimCLRProjectionHead(hid_dim, 32, 128)
        # SwaV: projection head and prototypes
        elif self.args.method == 'swav':
            self.projection_head = SwaVProjectionHead(hid_dim, 32, 128)
            self.prototypes = SwaVPrototypes(128, n_prototypes=32)
        self.ssl_training = ssl_training

    def conv_block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

    def forward(self, x, t=None):
        x = self.enc(x)
        x = torch.mean(x, dim=(2, 3))
        #print(x.shape,t.shape)
        if t is not None:
            x = torch.cat([x,t],axis=1)
            #print(x,t)
        if self.args.method == 'simclr' and self.ssl_training:
            return self.projection_head(x)
        elif self.args.method == 'swav' and self.ssl_training:
            x = self.projection_head(x)
            x = nn.functional.normalize(x, dim=1, p=2)
            return self.prototypes(x)
        else:
            return self.classifier(x)

class YearbookNetworkMOE(nn.Module):
    def __init__(self, args, num_input_channels, num_classes, ssl_training=False, t=None):
        super(YearbookNetworkMOE, self).__init__()
        self.args = args
        self.enc = nn.Sequential(self.conv_block(num_input_channels, 32), self.conv_block(32, 32),
                                 self.conv_block(32, 32), self.conv_block(32, 32))
        if args.incor_time:
            self.hid_dim = 33
            #self.classifier = nn.Linear(33, num_classes)
            self.classifier = MOENet_S(33,16,16,num_classes)
            #self.classifier = nn.Linear(33, num_classes)
        else:
            self.hid_dim = 32
            self.classifier = nn.Linear(32, num_classes)            
        # SimCLR: projection head
        if self.args.method == 'simclr':
            self.projection_head = SimCLRProjectionHead(hid_dim, 32, 128)
        # SwaV: projection head and prototypes
        elif self.args.method == 'swav':
            self.projection_head = SwaVProjectionHead(hid_dim, 32, 128)
            self.prototypes = SwaVPrototypes(128, n_prototypes=32)
        self.ssl_training = ssl_training

    def conv_block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

    def forward(self, x, t=None):
        x = self.enc(x)
        x = torch.mean(x, dim=(2, 3))
        #print(x.shape,t.shape)
        if t is not None:
            x = torch.cat([x,t],axis=1)
            #print(x,t)
        if self.args.method == 'simclr' and self.ssl_training:
            return self.projection_head(x)
        elif self.args.method == 'swav' and self.ssl_training:
            x = self.projection_head(x)
            x = nn.functional.normalize(x, dim=1, p=2)
            return self.prototypes(x)
        else:
            return self.classifier(x)
    
def _yearbook_init(args):
    if args.method in group_datasets:
        from data.yearbook import YearbookGroup
        dataset = YearbookGroup(args)
    else:
        from data.yearbook import Yearbook
        dataset = Yearbook(args)
    scheduler = None
    criterion = nn.CrossEntropyLoss(reduction=args.reduction).cuda()
    #network = YearbookNetwork2(configs, num_input_channels=3, num_classes=dataset.num_classes,t=True).cuda()#
    network = YearbookNetworkMOE(configs, num_input_channels=3, num_classes=dataset.num_classes,t=True).cuda()#
    optimizer = torch.optim.Adam(network.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    return dataset, criterion, network, optimizer, scheduler

def _fmow_init(args):
    if args.method in group_datasets:
        from data.fmow import FMoWGroup
        dataset = FMoWGroup(args)
    else:
        from data.fmow import FMoW
        dataset = FMoW(args)

    criterion = nn.CrossEntropyLoss(reduction=args.reduction).cuda()
    network = FMoWNetwork(args).cuda()
    optimizer = torch.optim.Adam((network.parameters()), lr=args.lr, weight_decay=args.weight_decay, amsgrad=True,
                                 betas=(0.9, 0.999))
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.96)
    return dataset, criterion, network, optimizer, scheduler

class DrugNet(nn.Module):
    def __init__(self, args, encoder, classifier):
        super(DrugNet, self).__init__()
        self.args = args
        self.enc = encoder
        self.classifier = classifier
    def forward(self, x, t=None):
        x = self.enc(x)
        if t is not None:
            x = torch.cat([x,t],axis=1)
        return self.classifier(x)

def DTI_Classifier2(in_features, out_features, is_nonlinear=False):
    if is_nonlinear:
        return torch.nn.Sequential(
            torch.nn.Linear(in_features, in_features // 2),
            torch.nn.ReLU(),
            torch.nn.Linear(in_features // 2, in_features // 4),
            torch.nn.ReLU(),
            torch.nn.Linear(in_features // 4, out_features))
    else:
        return torch.nn.Linear(in_features, out_features)
        
def _drug_init(args):
    if args.method in group_datasets:
        from data.drug import TdcDtiDgGroup
        dataset = TdcDtiDgGroup(args)
    else:
        from data.drug import TdcDtiDg
        dataset = TdcDtiDg(args)

    scheduler = None
    criterion = nn.MSELoss(reduction=args.reduction).cuda()
    featurizer = DTI_Encoder()
    if args.incor_time:
        classifier = DTI_Classifier2(featurizer.n_outputs+1, 1)
        network = DrugNet(args, featurizer, classifier)
    else:
        classifier = DTI_Classifier2(featurizer.n_outputs, 1)
        network = nn.Sequential(featurizer, classifier).cuda()
    optimizer = torch.optim.Adam(network.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    return dataset, criterion, network, optimizer, scheduler

def _mimic_init(args):
    if args.method in group_datasets:
        from data.mimic import MIMICGroup
        dataset = MIMICGroup(args)
    else:
        from data.mimic import MIMIC
        dataset = MIMIC(args)

    scheduler = None
    network = Transformer(args, embedding_size=128, dropout=0.5, layers=2, heads=2).cuda()
    class_weight = None
    if args.prediction_type == 'readmission':
        class_weight = torch.FloatTensor(np.array([0.26, 0.74])).cuda()
    elif args.prediction_type == 'mortality':
        if args.lisa:
            class_weight = torch.FloatTensor(np.array([0.03, 0.97])).cuda()
        else:
            class_weight = torch.FloatTensor(np.array([0.05, 0.95])).cuda()
    criterion = nn.CrossEntropyLoss(weight=class_weight, reduction=args.reduction).cuda()
    optimizer = torch.optim.Adam(network.parameters(), lr=args.lr)  # use lr = 5e-4
    return dataset, criterion, network, optimizer, scheduler

import torch.nn as nn
from transformers import DistilBertModel, DistilBertForSequenceClassification
class DistilBertClassifier(DistilBertForSequenceClassification):
    def __init__(self, config):
        super().__init__(config)

    def __call__(self, x):
        input_ids = x[:, :, 0]
        attention_mask = x[:, :, 1]
        outputs = super().__call__(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )[0]
        return outputs

class DistilBertFeaturizer(DistilBertModel):
    def __init__(self, config):
        super().__init__(config)
        self.d_out = config.hidden_size

    def __call__(self, x):
        input_ids = x[:, :, 0]
        attention_mask = x[:, :, 1]
        hidden_state = super().__call__(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )[0]
        pooled_output = hidden_state[:, 0]
        return pooled_output

class ArticleNetwork(nn.Module):
    def __init__(self, num_classes, t=None):
        super(ArticleNetwork, self).__init__()
        featurizer = DistilBertFeaturizer.from_pretrained("distilbert-base-uncased")
        self.featurizer = featurizer
        if t is not None:
            classifier = nn.Linear(featurizer.d_out+1, num_classes)
        else:
            classifier = nn.Linear(featurizer.d_out, num_classes)
        self.classifier = classifier
        #self.model = nn.Sequential(featurizer, classifier)
    def forward(self, x, t=None):
        x = self.featurizer(x)
        if t is not None:
            x = torch.cat([x,t],axis=1)
        return self.classifier(x)


class ArticleMOE(nn.Module):
    def __init__(self, num_classes, t=None, num_experts = 16):
        super(ArticleNetwork, self).__init__()
        featurizer = DistilBertFeaturizer.from_pretrained("distilbert-base-uncased")
        self.featurizer = featurizer
        if t is not None:
            classifier = MOENet_S(featurizer.d_out+1,512,16,num_classes)#nn.Linear(featurizer.d_out+1, num_classes)
        else:
            classifier = nn.Linear(featurizer.d_out, num_classes)
        self.classifier = classifier
    def forward(self, x, t=None):
        x = self.featurizer(x)
        if t is not None:
            x = torch.cat([x,t],axis=1)
        return self.classifier(x)

def _huffpost_init(args):
    if args.method in group_datasets:
        from data.huffpost import HuffPostGroup
        dataset = HuffPostGroup(args)
    else:
        from data.huffpost import HuffPost
        dataset = HuffPost(args)
    scheduler = None
    criterion = nn.CrossEntropyLoss(reduction=args.reduction).cuda()
    network = ArticleNetwork(num_classes=dataset.num_classes, t=args.incor_time).cuda()
    optimizer = torch.optim.AdamW(network.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    return dataset, criterion, network, optimizer, scheduler

def _arxiv_init(args):
    if args.method in group_datasets:
        from data.arxiv import ArXivGroup
        dataset = ArXivGroup(args)
    else:
        from data.arxiv import ArXiv
        dataset = ArXiv(args)
    scheduler = None
    criterion = nn.CrossEntropyLoss(reduction=args.reduction).cuda()
    network = ArticleNetwork(num_classes=dataset.num_classes,t=args.incor_time).cuda()
    optimizer = torch.optim.AdamW(network.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    return dataset, criterion, network, optimizer, scheduler

def trainer_init(args):
    random.seed(args.random_seed)
    np.random.seed(args.random_seed)
    torch.cuda.manual_seed(args.random_seed)
    torch.manual_seed(args.random_seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.cuda.set_device(args.device)
    if args.method in ['groupdro', 'irm']:
        args.reduction = 'none'
    else:
        args.reduction = 'mean'
    return globals()[f'_{args.dataset}_init'](args)

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
class MOENet(nn.Module):
    def __init__(self, input_dim, hidden, num_experts, output_dim = 2):
        super().__init__()
        self.num_experts = num_experts
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.hidden = hidden
        self.centers = nn.Parameter(torch.linspace(0, 1, num_experts))  #use a fixed start point, not good
        init = torch.full((num_experts,), 1.0)#1/2*num_experts)
        #self.temprature = nn.Parameter(torch.tensor(0.1))
        #self.register_buffer('log_sigma', torch.log(init))
        self.log_sigma = nn.Parameter(torch.log(init))
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(self.input_dim-1, hidden),#removing time feature from expert is better
                nn.ReLU(),
                nn.BatchNorm1d(hidden),
                torch.nn.Dropout(0.5),
                nn.Linear(hidden, hidden),
                nn.ReLU(),
                nn.BatchNorm1d(hidden),
                torch.nn.Dropout(0.5),
                nn.Linear(hidden, hidden),
                nn.ReLU(),
                nn.BatchNorm1d(hidden),
                torch.nn.Dropout(0.5),
                nn.Linear(hidden, output_dim),
            ) for _ in range(num_experts)
        ])
    def forward(self, x):
        timestamp = x[:, -1].unsqueeze(1) # (B,1)
        centers = self.centers.unsqueeze(0)  # (1,N)
        B = x.size(0)       
        #    w_ij = exp(- (t_j - c_i)^2 / (2σ^2) ) 
        sigma = torch.exp(self.log_sigma).unsqueeze(0) #(1,N)
        numerator = timestamp - centers
        route = torch.exp(- numerator.pow(2) / (2 * sigma.pow(2)))#/(sigma*torch.sqrt(2*torch.pi))  # (B, N)
        expert_outputs = torch.stack([expert(x[:,:-1]) for expert in self.experts], dim=1)
        output = torch.einsum('be,beo->bo', route, expert_outputs)
        return output


import torch
import torch.nn as nn
import torch.nn.functional as F
class MOENet_S(nn.Module):#smaller MoE
    def __init__(self, input_dim, hidden, num_experts, output_dim = 2):
        super().__init__()
        self.num_experts = num_experts
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.hidden = hidden
        self.centers = nn.Parameter(torch.linspace(0, 1, num_experts))  #use a fixed start point, not good
        init = torch.full((num_experts,), 1.0)
        self.log_sigma = nn.Parameter(torch.log(init))
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(self.input_dim-1, hidden),#removing time feature from expert is better
                nn.ReLU(),
                torch.nn.Dropout(0.5),
                nn.Linear(hidden, output_dim),
            ) for _ in range(num_experts)
        ])
    def forward(self, x):
        timestamp = x[:, -1].unsqueeze(1) # (B,1)
        centers = self.centers.unsqueeze(0)  # (1,N)
        B = x.size(0)       
        # 4) 计算 Gaussian 门控权重
        #    w_ij = exp(- (t_j - c_i)^2 / (2σ^2) ) 
        sigma = torch.exp(self.log_sigma).unsqueeze(0) #(1,N)
        numerator = timestamp - centers
        route = torch.exp(- numerator.pow(2) / (2 * sigma.pow(2)))#/(sigma*torch.sqrt(2*torch.pi))  # (B, N)
        expert_outputs = torch.stack([expert(x[:,:-1]) for expert in self.experts], dim=1)
        output = torch.einsum('be,beo->bo', route, expert_outputs)
        return output


In [3]:
#choosing the target dataset
#configs = argparse.Namespace(**configs_yearbook_erm)
#configs = argparse.Namespace(**configs_arxiv_erm)
configs = argparse.Namespace(**configs_huffpost_erm)
configs.train_update_iter = configs.train_update_iter*5
scheduler = None
group_datasets = ['coral', 'groupdro', 'irm']
print = partial(print, flush=True)
configs.incor_time=True
configs.random_seed=0
dataset, criterion, network, optimizer, scheduler = trainer_init(configs)
import os
#os.environ["CUDA_VISIBLE_DEVICES"] = "1,2,3"
#network = nn.DataParallel(network)
#mode 0 indicates training, mode 1 indicates val
#network = YearbookNetwork2(configs, num_input_channels=3, num_classes=dataset.num_classes,t=configs.incor_time).cuda()#
trainer = globals()[method_dict[configs.method]](configs, dataset, network, criterion, optimizer, scheduler)

Year 2012 loaded
Year 2013 loaded
Year 2014 loaded
Year 2015 loaded
Year 2016 loaded
Year 2017 loaded
Year 2018 loaded


In [6]:
configs

Namespace(dataset='yearbook', regression=False, prediction_type=None, method='erm', device=0, random_seed=0, train_update_iter=12000, lr=0.001, momentum=0.9, weight_decay=0.0, mini_batch_size=32, reduced_train_prop=None, eval_fix=True, difficulty=False, split_time=1970, eval_next_timestamps=10, load_model=False, eval_all_timestamps=False, K=1, lisa=False, lisa_intra_domain=False, mixup=False, lisa_start_time=0, mix_alpha=2.0, cut_mix=False, num_groups=10, group_size=5, non_overlapping=False, ewc_lambda=1.0, gamma=1.0, online=False, fisher_n=None, emp_FI=False, buffer_size=100, coral_lambda=1.0, irm_lambda=1.0, irm_penalty_anneal_iters=0, si_c=0.1, epsilon=0.001, ssl_finetune_iter=300, data_dir='./Data', log_dir='./checkpoints', results_dir='./results', num_workers=0, incor_time=True, reduction='mean')

In [4]:
trainer.train_dataset.datasets[2016][0]

{'title': array(['the high resolution infrared spectrum of hcl$^+$',
        'stability of markov regenerative switched linear systems',
        'the solution to a multichannel bethe potential and its application to\n  pion-nucleus reactions',
        ...,
        'a primer on carnot groups: homogenous groups, cc spaces, and regularity\n  of their isometries',
        'a master functional for quantum field theory',
        'dynamics of gravitating hadron matter in bianchi-ix cosmological model'],
       dtype='<U278'),
 'category': array([ 10,  66,  79, ..., 100,  80,  76])}

In [ ]:
#You can directly cap the time to 1 for easy implementation, and it will have similar performance as
# modifying the prediction algorithm of trainer.run() to enforce a difference of softmax output (just like we have done in core/nn.py)
#If you prefer to use the enforce, you can modify the 180 lines of wildtime/methods/base_trainer.py to apply it. 
#One thing to be noticed is that these are multi-classification tasks, you need to check if the probablility equals 1/C

In [8]:
#for yearbook
if configs.incor_time:
    print("Incorporate time")
    for ydata in trainer.train_dataset.datasets:
        #print(ydata)
        for mdata in trainer.train_dataset.datasets[ydata]:#(ydata-1930)/70
            trainer.train_dataset.datasets[ydata][mdata]['time'] = np.array([(ydata-1930)/40]*len(trainer.train_dataset.datasets[ydata][mdata]['labels']))
            trainer.eval_dataset.datasets[ydata][mdata]['time'] = np.array([(ydata-1930)/40]*len(trainer.eval_dataset.datasets[ydata][mdata]['labels']))
            trainer.eval_dataset.datasets[ydata][mdata]['time'][trainer.eval_dataset.datasets[ydata][mdata]['time']>1] = 1 
            #trainer.eval_dataset.datasets[ydata][mdata]['time'][trainer.eval_dataset.datasets[ydata][mdata]['time']>6] = 6 


Incorporate time


In [9]:
trainer.run()

Running Eval-Fix...

Saving model at timestamp 1970 to path ./checkpoints/yearbook_ERM-train_update_iter=24000-lr=0.001-mini_batch_size=32-seed=0-eval_fix_time=1970_T...


=================================== Results (Eval-Fix) ===================================
Metric: accuracy

ID accuracy: 	0.9886702444841979

OOD timestamp = 1971: 	 accuracy is 0.9040404040404041
OOD timestamp = 1972: 	 accuracy is 0.8702928870292888
OOD timestamp = 1973: 	 accuracy is 0.7603833865814696
OOD timestamp = 1974: 	 accuracy is 0.7919375812743823
OOD timestamp = 1975: 	 accuracy is 0.724053724053724
OOD timestamp = 1976: 	 accuracy is 0.6787709497206704
OOD timestamp = 1977: 	 accuracy is 0.6937269372693727
OOD timestamp = 1978: 	 accuracy is 0.7478632478632479
OOD timestamp = 1979: 	 accuracy is 0.7598116169544741
OOD timestamp = 1980: 	 accuracy is 0.6910299003322259
OOD timestamp = 1981: 	 accuracy is 0.7568627450980392
OOD timestamp = 1982: 	 accuracy is 0.7125506072874493
OOD timestamp = 1983: 	 ac

In [ ]:
print(trainer.train_dataset.datasets.keys())
print(trainer.eval_dataset.datasets.keys())

In [ ]:
#for article net
if configs.incor_time:
    print("Incorporate time")
    for ydata in trainer.train_dataset.datasets:
        print(ydata)
        for mdata in trainer.train_dataset.datasets[ydata]:#(ydata-1930)/70
            trainer.train_dataset.datasets[ydata][mdata]['time'] = np.array([(ydata-2007)/10]*len(trainer.train_dataset.datasets[ydata][mdata]['category']))
            trainer.eval_dataset.datasets[ydata][mdata]['time'] = np.array([(ydata-2007)/10]*len(trainer.eval_dataset.datasets[ydata][mdata]['category']))
            trainer.eval_dataset.datasets[ydata][mdata]['time'][trainer.eval_dataset.datasets[ydata][mdata]['time']>1] = 1
            #trainer.eval_dataset.datasets[ydata][mdata]['time'][trainer.eval_dataset.datasets[ydata][mdata]['time']>6] = 6 

In [ ]:
trainer.run()

In [ ]:
#for huffpost
if configs.incor_time:
    print("Incorporate time")
    for ydata in trainer.train_dataset.datasets:
        print(ydata)
        for mdata in trainer.train_dataset.datasets[ydata]:#(ydata-1930)/70
            trainer.train_dataset.datasets[ydata][mdata]['time'] = np.array([(ydata-2012)/3]*len(trainer.train_dataset.datasets[ydata][mdata]['category']))
            trainer.eval_dataset.datasets[ydata][mdata]['time'] = np.array([(ydata-2012)/3]*len(trainer.eval_dataset.datasets[ydata][mdata]['category']))
            trainer.eval_dataset.datasets[ydata][mdata]['time'][trainer.eval_dataset.datasets[ydata][mdata]['time']>1] = 1
            #trainer.eval_dataset.datasets[ydata][mdata]['time'][trainer.eval_dataset.datasets[ydata][mdata]['time']>6] = 6 

In [ ]:
trainer.run()

In [ ]:
#Refer to wild time paper, there is not a method demonstrates consistent improments across all datasets. but our strategies can be alway beneficial